# Deploying MiniMax Music 3 on SageMaker Async Inference

Async endpoint serving [MiniMax Music 3](https://huggingface.co/MiniMaxAI/MiniMax-Music3): lyrics + a
music description in, a 32 kHz 16-bit stereo WAV (up to 5 minutes) out. The model is an ~11B stack —
8B Qwen3 global LLM (autoregressive, 25 audio frames/s) + 0.6B local LLM + 2.4B flow-matching DiT +
Flow-VAE vocoder. bf16 weights are ~22 GB.

**Deployment shape** (same pattern as the FLUX.2 [klein] and LongCat endpoints in this repo):
- Weights live as **uncompressed objects** under an S3 prefix. SageMaker mirrors them to `/opt/ml/model/`. No tarball, no re-archiving on redeploy.
- The handler (`code/inference.py` + `code/requirements.txt`) is uploaded into a `code/` subfolder **inside the same prefix**. Edit the handler, re-upload two small files, redeploy — never touch ~29 GB of weights.
- We do **NOT** pass `entry_point`/`source_dir` to `PyTorchModel`: with dict `model_data` those trigger the SDK repack path, which silently skips wiring the code and you get the default handler's `Exactly one .pth or .pt file is required` crash.
- No `HF_TOKEN` in the container env. The Hub is only touched once during weight prep (and the repo is public anyway).

**Why `ml.g6e.2xlarge` (L40S 48 GB), fully GPU-resident, async:**
- bf16 weights ~22 GB, and classifier-free guidance is baked into the model (no off switch) — every request runs **two decode rows**, doubling the KV cache. A full 5-minute song costs ~3 GB of KV on top of the weights, so 24 GB cards (g5/g6) are out.
- **No CPU offload.** Offload is fine for a 4-step image model; here the global LLM runs thousands of *sequential* decode steps per song, so per-step PCIe swaps would multiply an already minutes-long generation. `inference.py` calls `pipe.to("cuda")`, never `enable_model_cpu_offload()`.
- A song takes **minutes** to generate — a real-time endpoint's 60 s invocation limit can't survive it. Async (S3 in / S3 out) is the only fit.

**Two traps specific to this model (already handled, don't undo):**
1. The `MiniMaxMusic3` pipeline is **not in any diffusers release** (verified against the 0.39.0 wheel). `requirements.txt` pins the exact PR commit from the model card as an immutable GitHub tarball. When [huggingface/diffusers#14456](https://github.com/huggingface/diffusers/pull/14456) ships in a release, switch to `diffusers==<release>`.
2. A 5-minute stereo WAV is ~38 MB — above TorchServe's ~6.5 MB default response cap. The deploy cell sets `TS_MAX_RESPONSE_SIZE`; removing it makes long songs fail opaquely.

**Sequence:**
1. One-time: run `prepare_weights.py` to download the diffusers components (~29 GB — it deliberately skips the ~29 GB of SGLang-only duplicates in the repo).
2. One-time: `aws s3 sync` those weights to a stable S3 prefix.
3. Per deploy: this notebook builds the `PyTorchModel` with `model_data` pointing at the prefix.


---

## ⚖️ License & usage disclaimer — read before running

**This notebook** is published for **demonstration and educational purposes** (demos, blog posts,
learning). It is provided as-is, without warranty of any kind.

**The model** it deploys, [MiniMax Music 3](https://huggingface.co/MiniMaxAI/MiniMax-Music3), is
governed by the **[MiniMax-Music3 Community License](https://huggingface.co/MiniMaxAI/MiniMax-Music3/blob/main/LICENSE)**.
By running this notebook you accept that license. The conditions that matter in practice:

- **Attribution**: any commercial product or service using the model must prominently display
  "MiniMax-Music3" on its user interface. (Recommended for demos too.)
- **Revenue cap**: use in products generating over **US $20M/year** in aggregate revenue requires
  prior written authorization from MiniMax (api@minimax.io).
- **Hosted generation**: if you expose the endpoint to third parties who generate their own outputs,
  you are required to implement and maintain reasonable safeguards against infringing use.
- **Acceptable Use Policy** (Exhibit A of the license) applies in full. Highlights:
  - Do **not** generate content that infringes third-party intellectual property — this includes
    using **copyrighted song lyrics** as input. All lyrics in this notebook are original.
  - Content shared publicly (blogs, social media, demos published online) must be **clearly
    disclosed as AI-generated**.
  - No impersonation of real artists or people without consent.
- Generated **outputs are your responsibility** — the license places the risk of use and
  distribution of outputs on you.

**Infrastructure costs are yours**: this notebook creates a SageMaker async endpoint on
`ml.g6e.2xlarge` (billed per hour while running). Run the Cleanup section when done.

*This summary is not legal advice. Read the full license before any use beyond personal
demos and educational content.*

---


## 1. Pin the SageMaker SDK

Pinned to v2 on purpose: this deploy pattern (dict `model_data` over an uncompressed S3 prefix,
handler auto-discovered from `code/` inside the prefix, no `entry_point`) is proven on v2's
`PyTorchModel`. SDK v3 replaces it with `ModelBuilder`, a different API that has not been validated
for this pattern. v2 emits a `SageMakerV2DeprecationWarning` on deploy — it's informational, not an
error; the config cell below suppresses it.


In [ ]:
!pip uninstall -y sagemaker
!pip install "sagemaker<3.0.0"


## 2. Configuration

Single instance option on purpose: `ml.g6e.2xlarge` (L40S, 48 GB) on the **PyTorch 2.5 inference DLC**
— the same DLC as the FLUX.2 endpoint, but note the transformers pin here is **5.15.0**, not FLUX.2's 4.56.2: the diffusers PR branch requires `huggingface-hub>=1.23`, which transformers 4.x refuses (`<1.0`) — that exact conflict failed this endpoint's first deploy with a pip `ResolutionImpossible` and a ping-health-check death. 5.15.0 was verified against the wheel: hub 1.x accepted, Qwen3 present, and the `torch.float8_e8m0fnu` crash from early 5.x is now hasattr-guarded, so it imports fine on torch 2.5. g7e
(Blackwell) is still not deployable on `PyTorchModel` (no inference DLC with CUDA ≥ 12.8); see the
FLUX.2 notebook for the full explanation and the upgrade steps when AWS ships one.


In [ ]:
import os

# SDK v2 is a deliberate choice (see section 1); silence its deprecation
# warning. Must be set before sagemaker is imported.
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = sess.boto_region_name

# Stable prefix where the prepared weights live. Referenced by every deploy —
# don't put a timestamp in here.
weights_prefix = "minimax-music3/weights"
weights_s3_uri = f"s3://{bucket}/{weights_prefix}/"

instance_type = "ml.g6e.2xlarge"        # L40S 48 GB — see rationale above
framework_version = "2.5"               # PyTorch inference DLC (Ada sm_89 kernels)
py_version = "py311"

endpoint_name = "minimax-music3-g6e"

print(f"region:          {region}")
print(f"bucket:          {bucket}")
print(f"weights_s3_uri:  {weights_s3_uri}")
print(f"instance_type:   {instance_type}")
print(f"endpoint_name:   {endpoint_name}")


## 3. Stage the weights + code to S3 (run once for weights)

`prepare_weights.py` downloads **only the diffusers components** (~29 GB) — the Hub repo is ~58 GB but
half of it (`qwen_7B/`, `flowmatching_vae.pth`, `dav.pth`) is the SGLang-Omni serving path, which the
endpoint never reads and which would double the S3 bill and the cold start.

The repo is public — no `HF_TOKEN` needed. Skip the download + sync on subsequent deploys. When you
change `code/inference.py`, only re-run the `code/` upload cell.

Verify the prefix is ready:
```bash
aws s3 ls --recursive s3://<bucket>/minimax-music3/weights/ --human-readable --summarize
```
Expected: ~29 GB across `modular_model_index.json`, `condition_encoder/`, `language_model/`,
`rvq_depth_decoder/`, `scheduler/`, `tokenizer/`, `transformer/`, `vocoder/`, plus a small `code/` folder.


In [ ]:
# Run the prep script (one-time). hf_transfer speeds the ~29 GB download ~5x.
# Comment this out on subsequent deploys.

# !pip install -q hf_transfer
# !HF_HUB_ENABLE_HF_TRANSFER=1 python prepare_weights.py --out weights


In [ ]:
# Sync the prepared weights to S3. `aws s3 sync` is idempotent — re-running
# only uploads changed files, so recovery from a partial run is cheap.
# Comment this out on subsequent deploys (weights don't change).

# !aws s3 sync weights/ {weights_s3_uri}


In [ ]:
# Upload the handler into a code/ subfolder INSIDE the model prefix.
# Re-run this cell after editing code/inference.py — it's a few KB.
# The inference toolkit loads /opt/ml/model/code/inference.py.

!aws s3 cp code/inference.py {weights_s3_uri}code/inference.py
!aws s3 cp code/requirements.txt {weights_s3_uri}code/requirements.txt

print("\nContents of code/ prefix (container reads /opt/ml/model/code/):")
!aws s3 ls {weights_s3_uri}code/


## 4. Deploy the endpoint

**Key wiring (each line exists because its absence broke a previous endpoint):**
- `model_data` is a dict with `CompressionType: None` — SageMaker mirrors the prefix to `/opt/ml/model/` verbatim.
- No `entry_point`/`source_dir` (silently breaks with dict model_data — the SDK repack path skips code wiring).
- `SAGEMAKER_MODEL_SERVER_TIMEOUT` / `TS_DEFAULT_RESPONSE_TIMEOUT` = 3600: a 5-minute song takes minutes of generation; the default worker timeout would kill the request mid-song.
- `TS_MAX_RESPONSE_SIZE` = 100 MB: a 5-minute stereo WAV is ~38 MB vs TorchServe's ~6.5 MB default cap.
- Delete any existing endpoint first: redeploying onto a crash-looping endpoint name does NOT reliably replace the model.
- `container_startup_health_check_timeout=3600`: cold start = pip install (incl. the diffusers PR tarball build, ~1 min) + mirror ~29 GB from S3 + load to GPU. Expect ~8–12 minutes total.

**Latency expectation:** generation is autoregressive at 25 frames/s of audio with mandatory CFG
(two decode rows per pass). On the L40S expect roughly **3–8 minutes for a full 5-minute song**;
short 10–15 s auditions come back in well under a minute. This is an async/batch workload by nature.


In [ ]:
from sagemaker.pytorch import PyTorchModel
from sagemaker.async_inference import AsyncInferenceConfig

async_config = AsyncInferenceConfig(
    output_path=f"s3://{bucket}/minimax-music3-outputs/",
    failure_path=f"s3://{bucket}/minimax-music3-failures/",
    max_concurrent_invocations_per_instance=1,
)

# Uncompressed-prefix model data: mirrored to /opt/ml/model/ verbatim.
model_data = {
    "S3DataSource": {
        "S3Uri":           weights_s3_uri,
        "S3DataType":      "S3Prefix",
        "CompressionType": "None",
    }
}

model = PyTorchModel(
    role=role,
    model_data=model_data,        # prefix containing weights + code/ subfolder
    framework_version=framework_version,
    py_version=py_version,
    env={
        # Generous worker timeout: model_fn loads ~22 GB and a single song
        # can take minutes to generate.
        "SAGEMAKER_MODEL_SERVER_TIMEOUT": "3600",
        "TS_DEFAULT_RESPONSE_TIMEOUT":    "3600",
        # A 5-min stereo WAV is ~38 MB; TorchServe's default response cap is
        # ~6.5 MB. Without this, long songs fail with an opaque error that
        # only shows in CloudWatch.
        "TS_MAX_RESPONSE_SIZE":           "104857600",
        # Belt-and-suspenders: the container hardcodes /opt/ml/model/code,
        # but set these too so intent is explicit.
        "SAGEMAKER_PROGRAM":              "inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY":     "/opt/ml/model/code",
    },
)

# Clean slate: delete any existing endpoint/config of this name first.
import boto3
sm = boto3.client("sagemaker")
try:
    sm.delete_endpoint(EndpointName=endpoint_name)
    print(f"Deleted existing endpoint {endpoint_name}")
except sm.exceptions.ClientError:
    pass
try:
    sm.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f"Deleted existing endpoint config {endpoint_name}")
except sm.exceptions.ClientError:
    pass

predictor = model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    instance_type=instance_type,
    async_inference_config=async_config,
    container_startup_health_check_timeout=3600,
    model_data_download_timeout=3600,
)


### If the deploy hangs or fails

The cell above blocks until `InService` or `Failed`. Diagnose from status + logs, not by guessing:
- `Creating` = still working (cold start is ~8–12 min here). `Failed` + `FailureReason` = the real answer.
- **No log group at all** = the container never booted; the failure was in the managed bootstrap (download / capacity), before TorchServe.
- diffusers/transformers lazy-import wrappers print a generic `RuntimeError: Failed to import ...` — the real cause is the line **after** `look up to see its traceback):`.
- Healthy cold start log sequence: `transformers 5.15.0 is in the supported range.` → `Qwen3ForCausalLM import confirmed.` → `diffusers 0.40.0.dev0 with ModularPipeline confirmed.` → `Staged local model view at /tmp/minimax_music3_local` (the model mount is read-only, so the patched index + weight symlinks live in /tmp) → `Components loaded in bf16` → `Pipeline fully resident on CUDA`.


In [ ]:
# Tail container logs while the deploy is in flight (run from a second cell/terminal):
# !aws logs tail /aws/sagemaker/Endpoints/{endpoint_name} --follow --since 1h

# Inspect the failure reason if the endpoint went to Failed:
# !aws sagemaker describe-endpoint --endpoint-name {endpoint_name} --query '{Status:EndpointStatus,Reason:FailureReason}' --output text

# Find the real error behind a lazy-import wrapper:
# !aws logs tail /aws/sagemaker/Endpoints/{endpoint_name} --since 30m --format short | grep -A2 "look up to see its traceback"


## 5. Helpers for invoking the endpoint

Async endpoints don't take inline payloads: PUT the JSON to S3, call `invoke_endpoint_async`, then
poll the `OutputLocation`. Unlike the FLUX.2 helper, this one **also polls the FailureLocation** —
a failed generation would otherwise spin forever (song-length jobs make that an expensive way to
find out something crashed).


In [ ]:
import io
import json
import time

import boto3

s3 = boto3.client("s3")
sm_runtime = boto3.client("sagemaker-runtime")


def _split_s3_uri(uri):
    path = uri.replace("s3://", "")
    b, k = path.split("/", 1)
    return b, k


def invoke(payload: dict, label: str, timeout_s: int = 2400, poll_s: int = 10) -> bytes:
    """Upload payload to S3, invoke the async endpoint, poll for the WAV.

    Polls the failure path too: a crashed generation raises with the
    container's error message instead of hanging until timeout.
    """
    input_key = f"minimax-music3-inputs/{label}-{int(time.time() * 1000)}.json"
    s3.put_object(
        Bucket=bucket,
        Key=input_key,
        Body=json.dumps(payload).encode("utf-8"),
        ContentType="application/json",
    )
    print(f"[{label}] input uploaded to s3://{bucket}/{input_key}")

    response = sm_runtime.invoke_endpoint_async(
        EndpointName=endpoint_name,
        InputLocation=f"s3://{bucket}/{input_key}",
        ContentType="application/json",
    )
    output_uri = response["OutputLocation"]
    failure_uri = response.get("FailureLocation")
    print(f"[{label}] output expected at {output_uri}")

    out_bucket, out_key = _split_s3_uri(output_uri)
    fail_bucket, fail_key = (_split_s3_uri(failure_uri) if failure_uri else (None, None))

    started = time.time()
    while True:
        try:
            obj = s3.get_object(Bucket=out_bucket, Key=out_key)
            print(f"[{label}] done in {time.time() - started:.0f}s")
            return obj["Body"].read()
        except s3.exceptions.NoSuchKey:
            pass
        if fail_key:
            try:
                err = s3.get_object(Bucket=fail_bucket, Key=fail_key)["Body"].read()
                raise RuntimeError(f"[{label}] generation FAILED: {err[:2000].decode('utf-8', 'replace')}")
            except s3.exceptions.NoSuchKey:
                pass
        if time.time() - started > timeout_s:
            raise TimeoutError(
                f"[{label}] no result after {timeout_s}s. Endpoint may still be "
                f"processing (long songs take minutes) — check CloudWatch: "
                f"aws logs tail /aws/sagemaker/Endpoints/{endpoint_name} --since 15m"
            )
        elapsed = int(time.time() - started)
        if elapsed and elapsed % 60 < poll_s:
            print(f"[{label}] still generating... {elapsed}s")
        time.sleep(poll_s)


## 6. Audition a caption with a short clip (cheapest iteration loop)

Generation time scales with duration — a 15 s clip is the fastest way to iterate on a music
description before committing to a full-length render. Two contract rules that silently ruin
output if broken (the handler warns on the second):
- **Both fields are required**: `lyrics` (what gets sung, with `[Verse]`/`[Chorus]` tags) and `prompt` (genre, BPM, key, instrumentation, vocal character, production).
- **Section tags go on their own lines.** `"[Verse] Walking down the street"` keeps the tag and silently DROPS the lyric text; `"[Verse]\nWalking down the street"` works.

Requests are deterministic in the seed — same lyrics + prompt + seed + duration returns identical
audio. Change only the seed to get another take of the same song.


In [ ]:
from IPython.display import Audio, display

audition_payload = {
    "lyrics": "[Verse]\nMorning light is spilling through the curtain\nEvery quiet street is yours and mine\n[Chorus]\nSoftly the world begins to breathe",
    "prompt": (
        "Genre: acoustic pop. BPM: 96. Key: C major. Warm and intimate, building gently into the chorus. "
        "Vocals: soft female lead, close and breathy, light stacked harmonies in the chorus. "
        "Arrangement: fingerpicked guitar and soft piano; brushed drums and upright bass enter in the chorus."
    ),
    "audio_duration": 15.0,
    "seed": 7,
}

audition_wav = invoke(audition_payload, label="audition")
with open("audition.wav", "wb") as f:
    f.write(audition_wav)
print(f"Saved audition.wav ({len(audition_wav) / 1e6:.1f} MB)")
display(Audio(audition_wav))


## 7. Full song

Once the caption sounds right, render the full length. A 60 s song typically takes a few minutes on
the L40S; a full 5-minute song can take up to ~8. The polling helper prints progress once a minute.

Structured captions (Global Metadata / Vocal Details / Arrangement) give the most control — the
model was trained on descriptions covering global attributes, emotional progression, and
section-level arrangement, and it uses all of it.


In [ ]:
song_payload = {
    "lyrics": (
        "[Verse]\nWalking down the empty street at midnight\nStreetlights flicker like a broken dream\n"
        "I've got nothing but the sound of my own heartbeat\nEchoing through the silent concrete stream\n"
        "[Chorus]\nAnd I keep on walking\nTill the morning finds me\nLeave the night behind me\n"
        "[Bridge]\nEvery shadow knows my name\n"
        "[Outro]\n(instrumental)"
    ),
    "prompt": (
        "A melancholic lo-fi hip-hop track at 85 BPM in F minor: mellow Rhodes piano riff, soft vinyl "
        "crackle, dusty boom-bap drums with a laid-back swing, warm upright bass. Intimate bedroom "
        "production, gentle tape saturation, no bright cymbals."
    ),
    "audio_duration": 90.0,
    "seed": 42,
}

song_wav = invoke(song_payload, label="song", timeout_s=3600)
with open("song.wav", "wb") as f:
    f.write(song_wav)
print(f"Saved song.wav ({len(song_wav) / 1e6:.1f} MB)")
display(Audio(song_wav))


## 8. Instrumental

Lyrics are required and must be non-empty — ask for an instrumental in the prompt and keep the
lyric line minimal.


In [ ]:
instrumental_payload = {
    "lyrics": "[Intro]\n(instrumental)\n[Outro]\n(instrumental)",
    "prompt": (
        "A moody synthwave instrumental at 100 BPM in D minor: pulsing analog bass arpeggio, gated "
        "reverb drum machine, wide atmospheric pads, and a soaring lead synth melody. Retro 1980s "
        "production, heavy chorus effect, cinematic and nocturnal. No vocals."
    ),
    "audio_duration": 30.0,
    "seed": 3,
}

instr_wav = invoke(instrumental_payload, label="instrumental")
with open("instrumental.wav", "wb") as f:
    f.write(instr_wav)
display(Audio(instr_wav))


## 8b. Save and play as MP3

The endpoint deliberately returns lossless WAV — that's the master. Convert to MP3 client-side when
you want a small shareable file (a 4-minute song drops from ~30 MB to ~6 MB). `soundfile`'s bundled
libsndfile encodes MP3 directly; no ffmpeg needed. The inline player accepts MP3 bytes just like WAV.


In [ ]:
import io

import soundfile as sf


def wav_to_mp3(wav_bytes: bytes, out_path: str | None = None) -> bytes:
    """Convert endpoint WAV bytes to MP3. Returns the MP3 bytes; optionally saves them."""
    pcm, sr = sf.read(io.BytesIO(wav_bytes))
    buf = io.BytesIO()
    sf.write(buf, pcm, sr, format="MP3")
    mp3 = buf.getvalue()
    if out_path:
        with open(out_path, "wb") as f:
            f.write(mp3)
        print(f"Saved {out_path} ({len(mp3) / 1e6:.1f} MB, was {len(wav_bytes) / 1e6:.1f} MB WAV)")
    return mp3


# Example: convert the last generation and play it inline.
song_mp3 = wav_to_mp3(song_wav, "song.mp3")
display(Audio(song_mp3))


## 8c. Zero-to-hero: generate straight to MP3 (single self-contained cell)

Everything in one cell: dependency install, config, async invoke with failure/timeout handling,
WAV→MP3 conversion, save, and inline playback. No other cell needs to have run. Assumes only that
the endpoint is `InService` and the environment has AWS credentials.


In [ ]:
# ============================================================================
# MiniMax Music 3 -> MP3, zero to hero. Self-contained: run just this cell.
# ============================================================================

# --- dependencies (soundfile's wheel bundles libsndfile 1.2+ with MP3 codec;
#     no ffmpeg needed. boto3 is preinstalled on SageMaker images, pinned here
#     for portability to any Jupyter env.)
%pip install -q "soundfile>=0.12,<0.14" "boto3>=1.34"

import io
import json
import time

import boto3
import soundfile as sf
from IPython.display import Audio, display

# --- config (edit these three if your deployment differs) -------------------
REGION        = "us-east-1"
ENDPOINT_NAME = "minimax-music3-g6e"
BUCKET        = None   # None -> derive the SageMaker default bucket automatically

if BUCKET is None:
    _acct = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
    BUCKET = f"sagemaker-{REGION}-{_acct}"

s3 = boto3.client("s3", region_name=REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=REGION)

# --- the song ---------------------------------------------------------------
# Contract rules: BOTH fields required; section tags on their OWN lines.
# The caption follows the model's Structured Caption format
# (Global Metadata / Vocal Details / Arrangement) -- it is the strongest control.
payload = {
    "lyrics": (
        "[Intro]\n(instrumental)\n"
        "[Verse]\nSo near to me, however far you wander\n"
        "No distance ever dimmed this heart of mine\n"
        "[Pre-Chorus]\nEvery word I could not say\nYou already knew\n"
        "[Chorus]\nNothing else matters when you call my name\n"
        "Let the whole world judge us, we remain the same\n"
        "[Solo]\n(instrumental)\n"
        "[Chorus]\nNothing else matters, burning through the night\n"
        "Two voices rising, one undying light\n"
        "[Outro]\n(instrumental)"
    ),
    "prompt": (
        "Global Metadata: symphonic metal ballad fused with Arabic orchestral music and operatic "
        "crossover. Slow 6/8 around 69 BPM, E minor with hijaz-inflected melodic ornamentation. "
        "Intimate at the start, swelling into a vast dramatic finale. Cinematic, wide, dynamic "
        "production with natural hall reverb.\n"
        "Vocal Details: a duet -- dark gravelly male baritone leading the verses, operatic female "
        "soprano with Arabic-style melisma answering; they join in soaring harmony on every chorus.\n"
        "Arrangement: lone fingerpicked clean electric guitar opens, joined by ney flute and soft "
        "strings; oud and riq build through the pre-chorus; choruses erupt with distorted guitars, "
        "full orchestra, timpani and darbuka; the solo is a melodic electric guitar lead over brass "
        "swells; the outro dissolves back to the lone fingerpicked guitar."
    ),
    "audio_duration": 60.0,   # audition length; raise to 240-300 for a full song
    "seed": 42,
}

# --- async invoke: upload payload, fire, poll output AND failure paths ------
label = f"mp3-hero-{int(time.time() * 1000)}"
input_key = f"minimax-music3-inputs/{label}.json"
s3.put_object(Bucket=BUCKET, Key=input_key,
              Body=json.dumps(payload).encode("utf-8"),
              ContentType="application/json")

resp = sm_runtime.invoke_endpoint_async(
    EndpointName=ENDPOINT_NAME,
    InputLocation=f"s3://{BUCKET}/{input_key}",
    ContentType="application/json",
)

def _split(uri):
    b, k = uri.replace("s3://", "").split("/", 1)
    return b, k

out_bucket, out_key = _split(resp["OutputLocation"])
fail_loc = resp.get("FailureLocation")
fail_bucket, fail_key = _split(fail_loc) if fail_loc else (None, None)

print(f"Generating {payload['audio_duration']:.0f}s of audio on {ENDPOINT_NAME}...")
started, TIMEOUT_S = time.time(), 3600
while True:
    try:
        wav_bytes = s3.get_object(Bucket=out_bucket, Key=out_key)["Body"].read()
        print(f"Done in {time.time() - started:.0f}s ({len(wav_bytes) / 1e6:.1f} MB WAV)")
        break
    except s3.exceptions.NoSuchKey:
        pass
    if fail_key:
        try:
            err = s3.get_object(Bucket=fail_bucket, Key=fail_key)["Body"].read()
            raise RuntimeError(f"Generation FAILED: {err[:2000].decode('utf-8', 'replace')}")
        except s3.exceptions.NoSuchKey:
            pass
    if time.time() - started > TIMEOUT_S:
        raise TimeoutError(f"No result after {TIMEOUT_S}s -- check CloudWatch logs for {ENDPOINT_NAME}")
    elapsed = int(time.time() - started)
    if elapsed and elapsed % 60 < 10:
        print(f"  still generating... {elapsed}s")
    time.sleep(10)

# --- WAV -> MP3, save both, play inline --------------------------------------
pcm, sr = sf.read(io.BytesIO(wav_bytes))
mp3_buf = io.BytesIO()
sf.write(mp3_buf, pcm, sr, format="MP3")
mp3_bytes = mp3_buf.getvalue()

with open(f"{label}.wav", "wb") as f:
    f.write(wav_bytes)          # lossless master
with open(f"{label}.mp3", "wb") as f:
    f.write(mp3_bytes)          # shareable copy

print(f"Saved {label}.wav ({len(wav_bytes) / 1e6:.1f} MB) and "
      f"{label}.mp3 ({len(mp3_bytes) / 1e6:.1f} MB)")
display(Audio(mp3_bytes))


## 8d. Demo-video background music pack

The three styles people actually put under YouTube demo videos, as ready-made instrumental presets:

- **`corporate_upbeat`** — the classic product-demo / explainer bed: bright, optimistic, unobtrusive.
- **`lofi_chill`** — tutorial / coding-video background: mellow, steady, non-distracting.
- **`cinematic_rise`** — product-launch / trailer energy: builds from soft to triumphant.

All instrumental (lyrics stay minimal per the model contract — describe "no vocals" in the caption).
Background music wants LOW dynamic contrast and no attention-grabbing solos, so the captions say so
explicitly. Edit `SELECTED` to generate fewer, and `BG_DURATION` to match your video length
(requires the `invoke` helper from section 5).

Reminder per the model license: clips you publish under a video must be disclosed as AI-generated.


In [ ]:
BG_PRESETS = {
    "corporate_upbeat": (
        "Global Metadata: upbeat corporate background music for a technology product demo video. "
        "112 BPM, 4/4, C major. Optimistic, clean, energetic but unobtrusive -- designed to sit "
        "under a voiceover. Polished modern production, moderate loudness, LOW dynamic contrast, "
        "no sudden drops or builds that would distract from narration.\n"
        "Vocal Details: fully instrumental, no vocals. The lead melodic role is carried by a "
        "bright plucked electric guitar and marimba-like synth plucks.\n"
        "Arrangement: light four-on-the-floor kick with hand claps on the backbeat, warm round "
        "bass, plucked guitar riff, airy synth pads, occasional piano octaves and subtle glockenspiel "
        "accents. Sections vary texture gently -- pads thin out, a shaker enters -- but the groove "
        "and energy stay constant from start to finish. Ends on a clean resolved chord."
    ),
    "lofi_chill": (
        "Global Metadata: lo-fi chillhop background music for a coding tutorial video. 82 BPM, "
        "4/4, D minor. Relaxed, warm, softly nostalgic, completely non-distracting. Dusty analog "
        "production with gentle tape saturation and soft vinyl crackle, low dynamic range.\n"
        "Vocal Details: fully instrumental, no vocals. The lead melodic role is a mellow Rhodes "
        "electric piano.\n"
        "Arrangement: laid-back boom-bap drums with a soft swing, muted upright bass, mellow "
        "Rhodes chords with light jazz voicings, occasional muted trumpet phrases far back in the "
        "mix, rain-like ambient texture. The loop evolves slowly -- a countermelody appears, the "
        "drums drop for two bars and return -- never demanding attention. Fades out gently."
    ),
    "cinematic_rise": (
        "Global Metadata: inspiring cinematic background music for a product launch video. 100 BPM, "
        "4/4, D major. Emotional arc from intimate and hopeful to expansive and triumphant, "
        "peaking at roughly three quarters of the duration, then settling into a warm resolution. "
        "Wide cinematic production, natural hall reverb.\n"
        "Vocal Details: fully instrumental, no vocals. The lead melodic role moves from solo piano "
        "to full string section.\n"
        "Arrangement: opens with soft piano arpeggios and a low string pad; a pulsing eighth-note "
        "string ostinato and light percussion enter and build; French horns lift the theme; the "
        "peak adds full orchestra, driving toms and soaring violins; the outro strips back to the "
        "solo piano motif. Smooth continuous build, no hard drops."
    ),
}

SELECTED = ["corporate_upbeat", "lofi_chill", "cinematic_rise"]
BG_DURATION = 45.0   # seconds; match your video segment length (up to 300)

bg_results = {}
for name in SELECTED:
    payload = {
        "lyrics": "[Intro]\n(instrumental)\n[Outro]\n(instrumental)",
        "prompt": BG_PRESETS[name],
        "audio_duration": BG_DURATION,
        "seed": 7,   # change for another take of the same style
    }
    wav = invoke(payload, label=f"bg-{name}")
    with open(f"bg_{name}.wav", "wb") as f:
        f.write(wav)
    bg_results[name] = wav
    print(f"=== {name} ===")
    display(Audio(wav))


## 9. Redeploying after a code change

When you edit `code/inference.py`:
1. Re-run the `code/` upload cell in section 3 (`aws s3 cp` of two small files). Weights untouched.
2. Re-run section 4 to recreate the endpoint against the same prefix.

The container only reads the handler at startup, so an in-place `aws s3 cp` does not affect a
running endpoint — you must redeploy for code changes to take effect.

**When diffusers ships the pipeline in a release** (watch [PR #14456](https://github.com/huggingface/diffusers/pull/14456)):
replace the GitHub-tarball line in `code/requirements.txt` with `diffusers==<release>`, verify the
release actually contains the pipeline first (grep the wheel for `minimax` files — a plausible
version number is not evidence), re-upload, redeploy. That removes the last cold-start dependency
on github.com.


## 10. Cleanup

In [ ]:
predictor.delete_endpoint()
sagemaker.Session().delete_model(model.name)
